# P7 – Chẩn đoán DeBERTa-v3-base: có hết NaN khi không dùng fp16 không?

**Bối cảnh.** Ở notebook 04 (chạy trên Kaggle T4, fp16), DeBERTa-v3-base cho `train_loss = NaN` ngay từ epoch 1 ở mọi fold, nên dự đoán kẹt ở một lớp (macro-F1 B = 0,2134, đúng bằng mức majority). Kết quả đó bị đánh dấu `failed_nonfinite` và bị loại khỏi P3.

**Giả thuyết ban đầu:** fp16 có biên độ quá hẹp cho DeBERTa-v3 → tràn số. GPU máy nhà (RTX A2000) hỗ trợ **bf16**, nên thử lại ở đây.

**Kết quả chẩn đoán ngày 26/10 – giả thuyết fp16 bị BÁC BỎ, đã tìm ra nguyên nhân thật:**
- Lần chạy đầu với bf16 **vẫn NaN** ngay ở batch 3.
- Kiểm tra thêm: checkpoint `microsoft/deberta-v3-base` được lưu ở **fp16**, và transformers 5.x mặc định nạp model **đúng kiểu đã lưu** → trọng số DeBERTa bị train thẳng ở fp16 (không có bản fp32 gốc) → NaN, bất kể autocast là fp16 hay bf16. RoBERTa/twitter-roberta lưu ở fp32 nên không bị.
- **Đã sửa** `src/sqe/train.py`: nạp encoder với `dtype=torch.float32`. Với RoBERTa/twitter-roberta không có gì thay đổi (vốn đã là fp32).
- Notebook này chạy lại sau khi sửa: xem ô "Train 1 fold × 1 epoch" bên dưới.

**Bài thử (giữ đúng thiết kế ban đầu):** Subtask **B**, input `comment`, `max_length = 128`, train fold 1–4, chấm **fold 0**, **1 epoch**. Cấu hình còn lại giống hệt lần chạy lỗi (lr 2e-5, batch 8 × tích luỹ 2, warmup 10%).

**Tiêu chí ĐẠT:** (1) loss hữu hạn (không NaN, không báo `FloatingPointError`), **và** (2) macro-F1 B trên fold 0 > 0,2134.

**Môi trường:** kernel **Python (sqe-gpu)** (`C:\Users\ADMIN\venvs\sqe`). **Không** cài/hạ cấp `transformers` trong notebook: ô kiểm tra tokenizer bên dưới cho thấy bản 5.14.1 tách từ DeBERTa đúng y như SentencePiece gốc. Đổi phiên bản thư viện sẽ làm kết quả P3/P4 cũ không tái lập được.

Notebook này **không lưu model**, chỉ ghi một file JSON nhỏ vào `outputs/p7_deberta/`.

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
import transformers

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "sqe").is_dir())
sys.path.insert(0, str(ROOT / "src"))

print("Python     :", sys.version.split()[0], "|", sys.executable)
print("torch      :", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("Dự án      :", ROOT)
if "venvs\\sqe" not in sys.executable:
    print("⚠️ Không phải kernel Python (sqe-gpu) – hãy đổi kernel rồi chạy lại.")
assert torch.cuda.is_available(), "PyTorch chưa thấy GPU – cần kernel Python (sqe-gpu)."

C:\Users\ADMIN\venvs\sqe\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python     : 3.14.4 | C:\Users\ADMIN\venvs\sqe\Scripts\python.exe
torch      : 2.11.0+cu128 | CUDA: True
transformers: 5.14.1
Dự án      : C:\Users\Public\Documents\NĂM 3\NLP\Đồ án\SemEval2027_Task8


In [2]:
import sqe.train as train_module
from sqe.data import load_en
from sqe.train import DEFAULT, ENCODERS, encode, fit
from transformers import AutoTokenizer

df = load_en()
cfg = {**DEFAULT, **ENCODERS["deberta"], "input": "comment", "max_length": 128, "epochs": 1}
print("Cấu hình:", cfg)
probe_model = train_module.MultiHeadClassifier(cfg["model"], {"B": 3})
print("Kiểu trọng số encoder sau khi nạp:", next(probe_model.encoder.parameters()).dtype, "(phải là torch.float32)")
del probe_model

Cấu hình: {'model': 'microsoft/deberta-v3-base', 'input': 'comment', 'max_length': 128, 'lr': 2e-05, 'epochs': 1, 'batch_size': 8, 'accum': 2, 'warmup': 0.1, 'weight_decay': 0.01, 'dropout': 0.1, 'class_weight': False, 'loss_weights': None, 'grad_ckpt': False, 'seed': 42}


Kiểu trọng số encoder sau khi nạp: torch.float32 (phải là torch.float32)


## 1. Tokenizer có tách từ đúng không?
So token của tokenizer trong transformers với SentencePiece gốc (`spm.model` của microsoft/deberta-v3-base) trên vài câu thật trong dữ liệu.

In [3]:
import sentencepiece as spm
from huggingface_hub import hf_hub_download

tok = AutoTokenizer.from_pretrained(cfg["model"])
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(cfg["model"], "spm.model"))
print(type(tok).__name__, "| fast:", tok.is_fast, "| vocab:", tok.vocab_size)

sample = df.yt_comment.sample(200, random_state=42).tolist()
same = sum(tok.convert_ids_to_tokens(tok(s)["input_ids"])[1:-1] == sp.encode(s, out_type=str) for s in sample)
print(f"Khớp SentencePiece gốc: {same}/{len(sample)} câu")
print("Ví dụ:", tok.convert_ids_to_tokens(tok(sample[0])["input_ids"])[:15])

DebertaV2Tokenizer | fast: True | vocab: 128000
Khớp SentencePiece gốc: 191/200 câu
Ví dụ: ['[CLS]', '▁People', '▁like', '▁Cody', '▁restore', '▁my', '▁faith', '▁in', '▁humanity', '.', '▁God', '▁bless', '▁your', '▁heart', '▁bro']


## 2. Chọn độ chính xác số: bf16 nếu GPU hỗ trợ, nếu không thì fp32 (tuyệt đối không fp16)
`fit()` trong `src/sqe/train.py` gọi `_amp_dtype()` để chọn kiểu số. Ta thay hàm đó **chỉ trong notebook này**, không sửa file nguồn, để các kết quả cũ giữ nguyên.

In [4]:
def _safe_amp_dtype(device):
    if device.type != "cuda":
        return None
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32


train_module._amp_dtype = _safe_amp_dtype
dev = torch.device("cuda")
PRECISION = str(_safe_amp_dtype(dev))
print("GPU:", torch.cuda.get_device_name(0), "| kiểu số dùng để train:", PRECISION)

GPU: NVIDIA RTX A2000 Laptop GPU | kiểu số dùng để train: torch.bfloat16


## 3. Train 1 fold × 1 epoch

In [5]:
ids = encode(tok, df, cfg)
tr, va = np.where(df.fold != 0)[0], np.where(df.fold == 0)[0]
print("train:", len(tr), "| valid (fold 0):", len(va))

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
status, hist, err = "ok", [], ""
try:
    model, hist = fit(df, tr, cfg, ("B",), ids, tok, dev, eval_idx=va)
    del model
except FloatingPointError as e:          # fit() tự dừng khi loss không hữu hạn
    status, err = "nonfinite", str(e)
minutes = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 2**30
torch.cuda.empty_cache()
print(f"Trạng thái: {status} | {minutes:.1f} phút | GPU tối đa {peak_gb:.2f} GB")
print(hist or err)

train: 2390 | valid (fold 0): 599


    epoch 1: loss 1.0475 | val macro-F1 B 0.2134
Trạng thái: ok | 2.5 phút | GPU tối đa 4.02 GB
[{'epoch': 1, 'train_loss': 1.0475222926873426, 'val_f1_B': np.float64(0.2133938706015891)}]


## 4. Kết luận tự động + so với các model khác ở cùng điểm (fold 0, sau epoch 1)
Các model khác được train 4 epoch nên đây chỉ là mốc tham khảo ở **epoch 1**, không phải so sánh cuối cùng.

In [6]:
MAJORITY_B = 0.2134
loss_ok = status == "ok" and all(np.isfinite(h["train_loss"]) for h in hist)
f1 = hist[-1]["val_f1_B"] if hist else float("nan")
passed = bool(loss_ok and f1 > MAJORITY_B)

ref = {}
for name in ["roberta_comment", "twhate_comment"]:
    meta = json.loads((ROOT / "outputs" / "runs" / f"{name}_B.json").read_text(encoding="utf-8"))
    ref[name] = meta["history"]["0"][0]["val_f1_B"]
print("Mốc epoch 1, fold 0:", {k: round(v, 4) for k, v in ref.items()})
print(f"DeBERTa ({PRECISION}): loss hữu hạn = {loss_ok}, macro-F1 B fold 0 = {f1:.4f}")
print("➡️  KẾT LUẬN (1 epoch):", "ĐẠT" if passed else
      ("loss đã hữu hạn nhưng chưa vượt majority → xem bài thử 4 epoch ở mục 5" if loss_ok else "CHƯA ĐẠT – loss NaN"))

out = ROOT / "outputs" / "p7_deberta"
out.mkdir(parents=True, exist_ok=True)
(out / "diag_fold0_ep1_B.json").write_text(json.dumps({
    "status": status, "passed": passed, "precision": PRECISION, "gpu": torch.cuda.get_device_name(0),
    "transformers": transformers.__version__, "torch": torch.__version__, "cfg": cfg,
    "history": hist, "error": err, "minutes": round(minutes, 2), "peak_gpu_gb": round(peak_gb, 2),
    "reference_epoch1_fold0": ref, "majority_B": MAJORITY_B,
    "note": "Chẩn đoán 1 fold x 1 epoch; KHÔNG phải điểm CV, không đưa vào all_results/P3."},
    ensure_ascii=False, indent=1), encoding="utf-8")
print("Đã ghi", out / "diag_fold0_ep1_B.json")

Mốc epoch 1, fold 0: {'roberta_comment': 0.3879, 'twhate_comment': 0.5051}
DeBERTa (torch.bfloat16): loss hữu hạn = True, macro-F1 B fold 0 = 0.2134
➡️  KẾT LUẬN (1 epoch): loss đã hữu hạn nhưng chưa vượt majority → xem bài thử 4 epoch ở mục 5
Đã ghi C:\Users\Public\Documents\NĂM 3\NLP\Đồ án\SemEval2027_Task8\outputs\p7_deberta\diag_fold0_ep1_B.json


## 5. Thử đúng cấu hình thật: 4 epoch, vẫn chấm fold 0
Bài thử 1 epoch ở trên nén cả lịch learning rate (warmup rồi giảm về 0) vào 1 epoch, nên chưa công bằng khi so với epoch 1 của RoBERTa (vốn nằm trong lịch 4 epoch). Ở đây chạy đúng cấu hình đã dùng cho mọi model khác (4 epoch) và xem đường cong từng epoch.

**Tiêu chí ĐẠT:** loss hữu hạn **và** macro-F1 B ở epoch 4 vượt rõ mức majority (> 0,30). So thêm với điểm cuối (epoch 4, fold 0) của RoBERTa / twitter-roberta để biết DeBERTa có đáng chạy đủ 5 fold không.

In [7]:
cfg4 = {**cfg, "epochs": 4}
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
status4, hist4, err4 = "ok", [], ""
try:
    model, hist4 = fit(df, tr, cfg4, ("B",), ids, tok, dev, eval_idx=va)
    del model
except FloatingPointError as e:
    status4, err4 = "nonfinite", str(e)
minutes4 = (time.time() - t0) / 60
torch.cuda.empty_cache()
ref_final = {}
for name in ["roberta_comment", "twhate_comment"]:
    meta = json.loads((ROOT / "outputs" / "runs" / f"{name}_B.json").read_text(encoding="utf-8"))
    ref_final[name] = meta["history"]["0"][-1]["val_f1_B"]
f1_4 = hist4[-1]["val_f1_B"] if hist4 else float("nan")
passed4 = bool(status4 == "ok" and all(np.isfinite(h["train_loss"]) for h in hist4) and f1_4 > 0.30)
print(f"Trạng thái: {status4} | {minutes4:.1f} phút")
for h in hist4:
    print(f"  epoch {h['epoch']}: loss {h['train_loss']:.4f} | macro-F1 B fold 0 {h['val_f1_B']:.4f}")
print("Mốc epoch 4, fold 0:", {k: round(v, 4) for k, v in ref_final.items()})
print("➡️  KẾT LUẬN (4 epoch):", "ĐẠT – đáng chạy đủ 5 fold × A, B" if passed4 else "CHƯA ĐẠT")
(out / "diag_fold0_ep4_B.json").write_text(json.dumps({
    "status": status4, "passed": passed4, "precision": PRECISION, "cfg": cfg4, "history": hist4,
    "error": err4, "minutes": round(minutes4, 2), "reference_epoch4_fold0": ref_final,
    "note": "Chẩn đoán fold 0, 4 epoch; KHÔNG phải điểm CV 5 fold, không đưa vào all_results/P3."},
    ensure_ascii=False, indent=1), encoding="utf-8")

    epoch 1: loss 1.0467 | val macro-F1 B 0.4348


    epoch 2: loss 0.8686 | val macro-F1 B 0.5939


    epoch 3: loss 0.6410 | val macro-F1 B 0.6175


    epoch 4: loss 0.4543 | val macro-F1 B 0.6338
Trạng thái: ok | 9.5 phút
  epoch 1: loss 1.0467 | macro-F1 B fold 0 0.4348
  epoch 2: loss 0.8686 | macro-F1 B fold 0 0.5939
  epoch 3: loss 0.6410 | macro-F1 B fold 0 0.6175
  epoch 4: loss 0.4543 | macro-F1 B fold 0 0.6338
Mốc epoch 4, fold 0: {'roberta_comment': 0.6171, 'twhate_comment': 0.6095}
➡️  KẾT LUẬN (4 epoch): ĐẠT – đáng chạy đủ 5 fold × A, B


1007

## 6. Kết luận P7 (26/10/2026)

| Lần thử | Kết quả |
|---|---|
| bf16 autocast, code cũ | **vẫn NaN** ở batch 3 → giả thuyết "do fp16" bị bác bỏ |
| Nguyên nhân thật | checkpoint `deberta-v3-base` lưu ở **fp16**; transformers 5.x nạp nguyên kiểu đó → trọng số bị train thẳng ở fp16 |
| Sửa `src/sqe/train.py`: nạp encoder `dtype=torch.float32` | loss hữu hạn. RoBERTa/twitter-roberta vốn fp32 nên không đổi; 43/43 test đạt |
| 1 fold × 1 epoch (lịch lr bị nén) | loss 1,047, vẫn đoán toàn majority (0,2134) — bài thử quá ngắn, chưa kết luận được |
| **1 fold × 4 epoch (cấu hình thật)** | macro-F1 B fold 0: 0,435 → 0,594 → 0,618 → **0,634**; ~9,5 phút trên RTX A2000 (bf16), GPU ~4 GB |

Mốc cùng fold 0, epoch 4: RoBERTa 0,617 · twitter-roberta-hate 0,610. **DeBERTa ĐẠT tiêu chí chẩn đoán**, nhưng đây mới là **1 fold**: chưa phải điểm CV, chưa được đưa vào P3.

**Vấn đề phụ đã phát hiện:** tokenizer fast của DeBERTa trong transformers 5 không chuẩn hoá Unicode như SentencePiece gốc → "…", ký tự nối emoji thành `[UNK]` ở 146/2989 comment (4,9%); chuẩn hoá NFKC trước khi tokenize còn 104. RoBERTa không bị (0 `[UNK]`). Nên xử lý trước khi chạy đủ 5 fold.

**Bước tiếp theo:** chạy đủ 5 fold × A, B (khoảng 95 phút trên máy nhà, hoặc Kaggle T4 với fp16 autocast nay đã có trọng số fp32) → OOF mới → chạy lại P3 bằng CPU.